# Readout probes: what can be read from the frozen PFN's state at BO contexts?

## Description

<!-- Written 2026-11-11 before the first run (pre-registration). Design: research_log/decisions/0006 §1. Issue #13. -->

**Question.** The round-1 value head (notebook 06) could not rank candidates, although clairvoyant headroom exists. It read the PFN's last-layer *query* states plus hand features. 0006 proposes a decoder that reads the context tokens' residual states at *every* layer. Before building it, small probes check whether the information is there:
- Is it readable from the frozen PFN's state at the BO contexts where the headroom sits?
- Do all layers carry more of it than the last layer, and more than the hand features?

**Setup.**
- *States*: the 4000 labeled states of `api_round1/pilot_20261111-133311` (split 4, d = 2, one per task, branch steps uniform over T = 20, 80% from EI rollouts). Pinned by sha256.
  - Probes are fit on 80% of the states and evaluated on the held-out 20%. Every state is a different task, so the split is by task.
- *Inputs*:
  - **hand**: the round-1 head's global features, plus its per-candidate features for candidate targets;
  - **last**: the last layer's context states, pooled (mean, max, and the incumbent's token), plus the candidate's last-layer query state;
  - **all**: the same at the embedding and after each of the 8 blocks.
- *Probes*: ridge regression (regularization chosen on held-out training tasks) and a one-hidden-layer MLP (256 units, early stopping). Small on purpose: they test readability, not modelling.
- *State targets* (ground truth from the task redrawn from its seed):
  - the remaining gap of the incumbent to f\* (% of the task's gap);
  - x\* (location);
  - the number of relevant inputs (from the prior);
  - roughness (log mean gradient norm);
  - the clairvoyant headroom of the state (cross-fitted best label advantage), and whether it exceeds 0.5 % of the gap (about the top 10%: "deviating pays here").
- *Candidate targets* (the 15 labeled non-base candidates per state):
  - the ground-truth advantage label;
  - one-step KG by re-encoding the PFN (8 stratified fantasies; a model quantity that tests whether lookahead information is linearly or shallowly readable).
- *Ceiling*: the split-half reliability of the advantage labels: the within-state Spearman between the labels from continuation replicates {0, 1} and {2, 3}. No probe can rank candidates better than the labels agree with themselves.
- *Metrics*: R² for regression; AUC for headroom_bin; mean within-state Spearman for candidate targets. Each comes with a bootstrap 95% CI over test states, and the contrasts all − last, all − hand and last − hand are paired bootstraps.

**What to look for.**
- The decision-relevant targets: headroom_bin and adv.
- The joint targets: gap_inc and x\*, as 0006 §1 sets out. If last-layer probes fail and all-layer probes succeed, the all-layer readout is confirmed. If both fail on joint targets, that argues for α-PFN information features or a joint head.
- KG readability.

**GO / NO-GO** (rule `analysis/api.py::probe_verdict`):
- **GO** (build the 0006 decoder on all layers): headroom_bin (AUC) or adv (within-state Spearman) is readable from all layers (CI lower bound above chance) *and* all layers beat the hand features (paired CI > 0).
- **NO-GO**: neither is readable from any PFN input. The decoder would then have nothing to read; the next levers are the belief itself (the model-based gate, notebook 07) or PFN fine-tuning (0004/0005).
- **INCONCLUSIVE**: otherwise.

**Limits.**
- One PFN (`pfn_large`, noise-free P0), d = 2, contexts n ≤ 24.
- 4000 states.
- The label-based targets are noisy (4 continuations); the reliability ceiling says how noisy.
- Probes are deliberately small. "Not readable by a small probe" is weaker than "not in the state".


In [ ]:
from anytime_acq.analysis import load_snapshot, paired_diff, show_header, summarize

SNAPSHOT = "results/api_probe/<sweep_id>"  # set after `just snapshot runs/api_probe/<sweep_id>`
snap = load_snapshot(SNAPSHOT)
show_header(snap)  # how it was run + configuration, generated from the snapshot

## Results

In [ ]:
# Aggregates use DONE members only; n_failed / n_started / n_missing say what is not in them yet.
summarize(snap.runs, "<metric>")

## Verdict

The GO / NO-GO criterion from the description, evaluated, in one line. Record it with `just decide`.